# Steps 5–6: Agent orchestrator and baselines

- **Agent:** OpenAI function calling over classifier, trends, and retrieval tools
- **Classifier baseline:** structured fields only
- **RAG baseline:** retrieval + single LLM call

Requires `OPENAI_API_KEY` in `.env`. Without a key, run classifier/trend examples only.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from src.agent.logging_utils import RunLogger
from src.agent.orchestrator import MSHASafetyAgent
from src.baselines.classifier_baseline import ClassifierBaseline
from src.baselines.rag_baseline import RAGBaseline
from src.tools.trends import TrendAnalyzer

HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY set: {HAS_KEY}")

In [ ]:
# Trend tool demo (no API key needed)
analyzer = TrendAnalyzer()
result = analyzer.count_by_year({"degree_code": "01"})
result.data[result.data["CAL_YR"] == 2015]

In [ ]:
# Classifier baseline demo
cls_q = (
    "Predict injury degree for: subunit_cd=03, classification_cd=05, "
    "occupation_cd=452010, activity_cd=05, injury_source_cd=2080, "
    "nature_injury_cd=010, inj_body_part_cd=410, mining_equip_cd=UNK, "
    "coal_metal_ind=C, accident_type_cd=01."
)
ClassifierBaseline().answer(cls_q)["answer"]

In [ ]:
if HAS_KEY:
    agent = MSHASafetyAgent()
    logger = RunLogger("notebook_agent")
    out = agent.answer("How many MSHA fatalities occurred in 2015?", logger=logger)
    print(out["answer"])
    print("Tools used:", out["tools_used"])
else:
    print("Set OPENAI_API_KEY in .env to run the agent and RAG baseline.")